In [ ]:
import pandas as pd
import openpyxl
from openpyxl import load_workbook
import datetime
import csv
import numpy as np
import time
from datetime import datetime
start_timehehe = datetime.now()

# SQL CONNECTION

In [ ]:
'''
%pip install sqlalchemy pymysql pandas
%pip install pyodbc sqlalchemy
'''

import os
from sqlalchemy import create_engine

# SQL Server connection string
server = 'ashley-edw.database.windows.net'
database = 'ASHLEY_EDW'
username = os.getenv("EDW_USERNAME")
password = 'your_password'

from urllib.parse import quote_plus

# SQL Server connection string
server = 'ashley-edw.database.windows.net'
database = 'ASHLEY_EDW'
# Using Microsoft Entra ID (Azure AD) integrated authentication
# No username/password needed - uses your Azure AD credentials

# Connection string for Azure AD authentication
connection_string = (
    f"mssql+pyodbc://{username}@{server}/{database}"
    f"?driver=ODBC+Driver+18+for+SQL+Server"
    f"&Authentication=ActiveDirectoryIntegrated"
    f"&Encrypt=yes" 
    f"&TrustServerCertificate=no"
)


# Create connection string for SQL Server
#connection_string = f'mssql+pymssql://{username}:{password}@{server}/{database}'

# Create engine
engine = create_engine(connection_string)



# ON HAND INVENTORY - NO USE

In [ ]:
'''
query = """

select 
    trim(ItemNumber) as ItemNumber,
    Warehouse,
    DateWeekEnding,
    OnHandQty,
    ItemStatus
from [Inventory_Enh_History].[ItemBalance]
where CAST(DateWeekEnding AS DATE) = (
    select max(CAST(DateWeekEnding AS DATE)) from [Inventory_Enh_History].[ItemBalance]
)
and Warehouse in ('1', '15', '17', '28', '335', '5', 'ECR')
"""

onhand = pd.read_sql(query, engine)
'''


# Lead time

Parent section. Children: Transit time, Supplier FOB, Vendor LT.

## Transit time

In [ ]:
transit_time = """select 
dt.dltwarehouse as whse,
dt.dltdeliverytime as delivery_days,
dt.dltfobcode as fob_code,
pm.pteCityOfPort as fob_port
from [Wholesale_ProductSourcing_AFI].[DeliveryTimes] dt
left join [Wholesale_ProductSourcing].[PortMaster] pm
    on dt.dltfobcode = pm.pteFOBPortCode"""

leadtime = pd.read_sql(transit_time, engine)
leadtime['weekly_lead_time'] = (leadtime['delivery_days'] / 7).apply(lambda x: round(x + 0.49))
leadtime

## Supplier FOB

In [ ]:
supplier_fob = """
Select
    V.VNDNR as [Vendor #],
    V.VNAME as [Vendor],
    V.VNAMA as [Office],
    V.FOBCD as [FOB Code],
    PM.pteShortPortDescription as [FOB]
from [Wholesale_Purchasing_AFI].[VENNAM] V
left join [Wholesale_ProductSourcing].[PortMaster] PM
    on V.FOBCD = PM.pteFobPortCode
where V.FOBCD <> ''
    AND V.VNDNR NOT LIKE 'A%';

"""

supplier_fob = pd.read_sql(supplier_fob, engine)
supplier_fob

In [ ]:
supplier_fob.to_clipboard(index=False)

## Vendor LT

In [ ]:
vendor_port_lt = supplier_fob.merge(leadtime, left_on='FOB Code', right_on='fob_code', how='left')
vendor_port_lt['Vendor #'] = vendor_port_lt['Vendor #'].astype(str).str.strip()
vendor_port_lt['whse'] = vendor_port_lt['whse'].apply(lambda x: pd.to_numeric(x, errors='coerce') if str(x).replace('.', '', 1).isdigit() else x)

vendor_port_lt.rename(columns={'Vendor #': 'Vendor','Vendor': 'Vendor Name'}, inplace=True)

vendor_port_lt.to_clipboard(index=False)


In [ ]:
"""
from turtle import left
import pandas as pd

vendor_warehouse_city_data = [
    {'Vendor': '600039', 'City': 'HO CHI MINH CITY'},
    {'Vendor': '900515', 'City': 'HO CHI MINH CITY'},
    {'Vendor': '900639', 'City': 'HO CHI MINH CITY'},
    {'Vendor': '659852', 'City': 'HO CHI MINH CITY'},
    {'Vendor': '624556', 'City': 'DA NANG'},
]

vendor_port = pd.DataFrame(vendor_warehouse_city_data)
vendor_port_lt = vendor_port.merge(leadtime, left_on='City', right_on='fob_port', how='left')
vendor_port_lt['Vendor'] = vendor_port_lt['Vendor'].astype(int)

vendor_port_lt['whse'] = vendor_port_lt['whse'].apply(lambda x: pd.to_numeric(x, errors='coerce') if str(x).replace('.', '', 1).isdigit() else x)

vendor_port_lt 
"""

# PURCHASE ORDER - NO USE


In [ ]:
'''

po = """ 
WITH PO_Summary AS (
    SELECT	
        pomordernum AS PO,
        CAST(pomcreated AS DATE) AS POCreateDate,
        CAST(pomoriginaletd AS DATE) AS OriginalETD,
        pomPOClass AS POType,
        pomvendornum AS Vendor,
        CAST(pometd AS DATE) AS ETD,
        CAST(pometa AS DATE) AS ETA,
        CAST(pomdue AS DATE) AS DueDate,
        CAST(pomSubmitToTreasury AS DATE) AS SubmitToTreasury,
        -- Normalize to week ending (Saturday)
        DATEADD(DAY, 7 - DATEPART(WEEKDAY, CAST(pometd AS DATE)), CAST(pometd AS DATE)) AS ETD_Week,
        DATEADD(DAY, 7 - DATEPART(WEEKDAY, CAST(pometa AS DATE)), CAST(pometa AS DATE)) AS ETA_PO,
        DATEADD(DAY, 7 - DATEPART(WEEKDAY, CAST(pomdue AS DATE)), CAST(pomdue AS DATE)) AS DueDate_week,
        DATEADD(DAY, 7 - DATEPART(WEEKDAY, CAST(pomSubmitToTreasury AS DATE)), CAST(pomSubmitToTreasury AS DATE)) AS SubmitToTreasury_week,    
        pomwarehouse AS Whse,
        pomstatuscode AS PO_Status,
        pomOrigCntrSize AS Mode
    FROM [Wholesale_ProductSourcing_AFI].[PoMaster]
    WHERE pomvendornum IN ('624556','600039','900515','900639')
      AND pomwarehouse IN ('335','1','ECR','15','17','28','5')
      AND pomstatuscode <> '99'
      AND CAST(pometd AS DATE) BETWEEN 
            (SELECT DATEADD(MONTH, -12, MAX(CAST(pometd AS DATE))) 
             FROM [Wholesale_ProductSourcing_AFI].[PoMaster]
             WHERE pomvendornum IN ('624556','600039','900515','900639')
               AND pomwarehouse IN ('335','1','ECR','15','17','28','5')
               AND pomstatuscode <> '99')
          AND
            (SELECT MAX(CAST(pometd AS DATE))
             FROM [Wholesale_ProductSourcing_AFI].[PoMaster]
             WHERE pomvendornum IN ('624556','600039','900515','900639')
               AND pomwarehouse IN ('335','1','ECR','15','17','28','5')
               AND pomstatuscode <> '99')
)

SELECT
    ps.*,
    pd.poditemnum AS Item,
    pd.podqtyordered AS QtyOrdered,
    pd.poditemclass AS ITCLS
FROM PO_Summary ps
LEFT JOIN [Wholesale_ProductSourcing_AFI].[PoDetail] pd
    ON ps.PO = pd.podordernum


"""

po = pd.read_sql(po, engine)
po


'''


In [ ]:
'''
po_clean = po.copy()
po_clean["ETD_Week"] = pd.to_datetime(po_clean["ETD_Week"], errors="coerce")
po_clean["ETA_PO"] = pd.to_datetime(po_clean["ETA_PO"], errors="coerce").fillna(pd.Timestamp("1900-01-01"))
po_group = (
    po_clean.groupby(
        ['PO', 'POType', 'Vendor',
         'ETD_Week', 'ETA_PO', 'DueDate_week','SubmitToTreasury_week','Whse', 'Mode', 'PO_Status', 'Item', 'ITCLS'],
        as_index=False
    )['QtyOrdered']
    .sum()
)

po_group['Vendor'] = po_group['Vendor'].apply(lambda x: pd.to_numeric(x, errors='coerce') if str(x).replace('.', '', 1).isdigit() else x)
po_group['Whse'] = po_group['Whse'].apply(lambda x: pd.to_numeric(x, errors='coerce') if str(x).replace('.', '', 1).isdigit() else x)

po_group = po_group.merge(vendor_port_lt, left_on=['Vendor', 'Whse'], right_on=['Vendor', 'whse'], how='left')
po_group['weekly_lead_time'] = pd.to_numeric(po_group['weekly_lead_time'], errors='coerce').fillna(0).astype(int)

mask_eta_default = po_group["ETA_PO"].eq(pd.Timestamp("1900-01-01"))
if mask_eta_default.any():
    eta_fill = (
        po_group.loc[mask_eta_default, "ETD_Week"]
        + pd.to_timedelta(po_group.loc[mask_eta_default, "weekly_lead_time"] * 7, unit="D")
    )
    po_group.loc[mask_eta_default, "ETA_PO"] = eta_fill.where(eta_fill.notna(), pd.Timestamp("1900-01-01"))

po_group['ETA'] = po_group['ETD_Week'] + pd.to_timedelta(po_group['weekly_lead_time'] * 7, unit='D')
'''



# PRODUCTION RESOURCES

In [ ]:
query2 = """

WITH Latest AS (
    SELECT ItemID, LocationID, ProdResourceID, [EndEffectiveDate], SnapshotDate
    FROM SupplyChain_Enh.ProductionConversion
    WHERE [SnapshotDate] = (
        SELECT MAX([SnapshotDate]) 
        FROM SupplyChain_Enh.ProductionConversion
    )
    AND [EndEffectiveDate] IS NOT NULL
    
    
),
AllProductionResource AS (
    SELECT ItemID, LocationID, ProdResourceID, [EndEffectiveDate], SnapshotDate,
    ROW_NUMBER() OVER (PARTITION BY ItemID, LocationID ORDER BY SnapshotDate DESC, EndEffectiveDate DESC) AS RowNumber
    FROM SupplyChain_Enh.ProductionConversion
    WHERE [EndEffectiveDate] IS NOT NULL
    
     
   

),
RecentSnapshot AS (
    SELECT *
    FROM AllProductionResource
    WHERE RowNumber = 1
)

SELECT 
    COALESCE(Latest.ItemID, RecentSnapshot.ItemID) AS [ItemID],
    COALESCE(Latest.LocationID, RecentSnapshot.LocationID) AS [LocationID],
    COALESCE(Latest.ProdResourceID, RecentSnapshot.ProdResourceID) AS [ProdResourceID],
    COALESCE(Latest.EndEffectiveDate, RecentSnapshot.EndEffectiveDate) AS [EndEffectiveDate],
    COALESCE(Latest.SnapshotDate, RecentSnapshot.SnapshotDate) AS [SnapshotDate]
FROM Latest
FULL OUTER JOIN RecentSnapshot 
    ON Latest.ItemID = RecentSnapshot.ItemID
    AND Latest.LocationID = RecentSnapshot.LocationID
"""

prod = pd.read_sql(query2, engine)



prod



In [ ]:

query3 = '''
select
TRIM([Item SKU]) AS [Item SKU],
CASE
    WHEN CHARINDEX('-', [Item SKU]) > 0
        THEN LEFT([Item SKU], CHARINDEX('-', [Item SKU]) - 1)
    ELSE LEFT([Item SKU], 5)
END AS [Series],
[Item Description],
[Series Number],
[Collective Class],
[Product Line],
[AFI Finance Division],
[Primary Vendor],
[Current Status],
[Future Status]
from [PowerBI_SupplyChain].[CurrentProductDetails]

'''


status = pd.read_sql(query3, engine)



In [ ]:

# prod = chuẩn; map status theo ItemID = Item SKU
status_cols = [
    "Series","Item SKU", "Collective Class",
    "AFI Finance Division", "Current Status", "Future Status",
]
status_map = status[status_cols].drop_duplicates(subset=["Item SKU"])
status_map["Item SKU"] = status_map["Item SKU"].astype(str).str.strip()
prod["ItemID"] = prod["ItemID"].astype(str).str.strip()

item_profile = prod.merge(
    status_map,
    how="left",
    left_on="ItemID",
    right_on="Item SKU",
)

item_profile["ItemID"] = item_profile["ItemID"].astype(str).str.strip()
item_profile = item_profile.drop_duplicates(subset=["ItemID"], keep="first")
item_profile = item_profile.fillna("")

print(f"prod rows: {len(prod)} | matched status: {item_profile['Current Status'].notna().sum()}")


item_profile

# MAKE BUY CODE

In [ ]:

query6 = '''
-- MB code tuần Monday mới nhất
WITH LatestMonday AS (
    SELECT MAX(CAST(LGL.[FileDate] AS date)) AS LatestFileDate
    FROM [SupplyChain_Enh].[ForecastCommonContainer_Logility] LGL
    INNER JOIN [Enterprise_DW].[DimDate] DD
        ON DD.[DateID] = CAST(LGL.[FileDate] AS date)
    WHERE DD.[CalendarDayOfWeekName] = 'Monday'
      
),

MostRecentDateResources AS (
    SELECT MAX([SnapshotDate]) AS LatestDate
    FROM SupplyChain_Enh.ProductionConversion
),

ProdResource AS (
    SELECT DISTINCT
        PC.ItemID,
        PC.ProdResourceID
    FROM SupplyChain_Enh.ProductionConversion PC
    CROSS JOIN MostRecentDateResources MRD
    WHERE PC.[SnapshotDate] = MRD.LatestDate
      AND PC.[EndEffectiveDate] IS NOT NULL
      AND CAST(PC.[EndEffectiveDate] AS date) >= CAST(GETDATE() AS date)
)

SELECT DISTINCT
    LGL.[Planning Vendor] AS Vendor_code,
    VM.[VendorName],
    VM.[Office],
    LGL.[ProductGroup] AS Series,
    LGL.[Item-Lvl1] AS SKU,
    LGL.[Collective Class],
    LGL.[Field1] AS Item_Status,
    LGL.[Field-19] AS Future_status,
    LGL.[Location],
    LGL.[Source],
    LGL.[MK_BuyCODE] AS [MB],
    PR.[ProdResourceID],
    CAST(LGL.[FileDate] AS date) AS [FileDate],
    DATEADD(
        DAY,
        (7 - DATEPART(WEEKDAY, LGL.[FileDate])) % 7,
        CAST(LGL.[FileDate] AS date)
    ) AS ETD,
    DD.[CalendarWeekIndicator]
FROM [SupplyChain_Enh].[ForecastCommonContainer_Logility] LGL
CROSS JOIN LatestMonday LM
LEFT JOIN [PowerBI_SupplyChain].[VendorMaster] VM
    ON VM.[VendorNumber] = LGL.[Planning Vendor]
LEFT JOIN [Enterprise_DW].[DimDate] DD
    ON DD.[DateID] = CAST(LGL.[FileDate] AS date)
LEFT JOIN ProdResource PR
    ON PR.ItemID = LGL.[Item-Lvl1]
WHERE CAST(LGL.[FileDate] AS date) = LM.LatestFileDate
  AND DD.[CalendarDayOfWeekName] = 'Monday'
ORDER BY
    LGL.[Planning Vendor],
    LGL.[Item-Lvl1],
    LGL.[Location];
'''


mb_code = pd.read_sql(query6, engine)



In [ ]:
for col in ['Vendor_code', 'VendorName', 'Office', 'Series', 'SKU',
            'Collective Class', 'Item_Status', 'Future_status', 'Location',
            'Sleeper', 'Source', 'MB', 'ProdResourceID', 'FileDate', 'ETD',
            'CalendarWeekIndicator']:
    if col in mb_code.columns:
        mb_code[col] = mb_code[col].astype(str).str.strip()



# DUAL SOURCE - PSW

In [ ]:
query5 = f"""
WITH PSW AS (
  SELECT 
      Item,
      Whse,
      Vendor,
      WeekNum,
      ITDESC,
      ITEMCLASS,
      VNAME,
      SQty,
      PQty,
      FQty,
      SPRunDate, 
      FORMAT(SPRunDate, 'yyyy-MM-dd') AS RunDate,
      CAST(DATEADD(day, 7 - DATEPART(WEEKDAY, SPRunDate), SPRunDate) AS date) AS SPRunETD,
      CAST(DATEADD(week, WeekNum, DATEADD(day, 7 - DATEPART(WEEKDAY, SPRunDate), SPRunDate)) AS date) AS ETD,
      DATENAME(WEEKDAY, SPRunDate) AS RunDay,
      ROW_NUMBER() OVER (
        PARTITION BY 
          Item,
          CAST(DATEADD(week, WeekNum, DATEADD(day, 7 - DATEPART(WEEKDAY, SPRunDate), SPRunDate)) AS date)
        ORDER BY SPRunDate DESC
      ) AS rn
    FROM [SupplyChain_Enh].[PSWWeeklyExtractSnapshot]
   WHERE CAST(SPRunDate AS date) = (
          SELECT MAX(CAST(SPRunDate AS date)) 
          FROM [SupplyChain_Enh].[PSWWeeklyExtractSnapshot])
),
MostRecentDateResources AS (
    SELECT MAX([SnapshotDate]) AS LatestDate
    FROM SupplyChain_Enh.ProductionConversion
),
ProdResource AS (
    SELECT ItemID, LocationID, ProdResourceID
    FROM SupplyChain_Enh.ProductionConversion
    CROSS JOIN MostRecentDateResources MRD
    WHERE [SnapshotDate] = MRD.LatestDate
    AND [EndEffectiveDate] IS NOT NULL
),
Prod_snapshot AS (
  SELECT
    psw.*,
    pr.LocationID,
    pr.ProdResourceID
  FROM PSW psw
  LEFT JOIN ProdResource pr
    ON psw.Item = pr.ItemID
   AND psw.Vendor = pr.LocationID

),
dual_source as(
  SELECT 
  [Item SKU],
  [Primary Vendor Number],
  [Primary Vendor Name],
  [Primary Vendor Split],
  [Secondary Vendor Number],
  [Secondary Vendor Name],
  [Secondary Vendor Split],
  [Snapshot date]
FROM SupplyChain_Enh.VendorSplits
WHERE [Snapshot date] = (
  SELECT MAX([Snapshot date]) 
  FROM SupplyChain_Enh.VendorSplits
)
AND [Primary Vendor Split] != 1.00
),
prosnap_dualsource AS (
  SELECT 
    ps.*,
    ds.[Primary Vendor Number],
    ds.[Primary Vendor Name],
    ds.[Primary Vendor Split],
    ds.[Secondary Vendor Number],
    ds.[Secondary Vendor Name],
    ds.[Secondary Vendor Split],
    CASE
      WHEN LTRIM(RTRIM(CAST(ps.Vendor AS varchar(50))))
         = LTRIM(RTRIM(CAST(ds.[Secondary Vendor Number] AS varchar(50))))
      THEN 'Yes'
      ELSE 'No'
    END AS DualSource
  FROM Prod_snapshot ps
  INNER JOIN dual_source ds
    ON ps.Item = ds.[Item SKU]
)
SELECT 
  Item,
  Whse,
  Vendor,
  VNAME,
  SQty,
  PQty,
  FQty,
  (SQty + FQty + PQty) as TotalQty,
  ETD,
  DualSource,
  [Primary Vendor Number],
  [Primary Vendor Name],
  [Primary Vendor Split],
  [Secondary Vendor Number],
  [Secondary Vendor Name],
  [Secondary Vendor Split]
FROM prosnap_dualsource
WHERE 
  DualSource = 'Yes'
  AND (SQty + FQty + PQty) > 0
  AND CAST(ETD AS DATE) > CAST(GETDATE() AS DATE)


"""
dual = pd.read_sql(query5, engine)
dual



In [ ]:
# Merge lead time using Vendor × Warehouse after normalizing both join keys.
def _merge_key(values):
    return (
        values.astype("string")
        .str.strip()
        .str.replace(r"\.0+$", "", regex=True)
    )

_dual = dual.copy()
_lt = vendor_port_lt[["Vendor", "whse", "weekly_lead_time"]].copy()

_dual["_vendor_key"] = _merge_key(_dual["Vendor"])
_dual["_whse_key"] = _merge_key(_dual["Whse"])
_lt["_vendor_key"] = _merge_key(_lt["Vendor"])
_lt["_whse_key"] = _merge_key(_lt["whse"])
_lt["weekly_lead_time"] = pd.to_numeric(
    _lt["weekly_lead_time"], errors="coerce"
)

# Remove exact duplicate mappings, but stop if one Vendor × Whse has
# conflicting lead-time values.
_lt = _lt.drop_duplicates(
    subset=["_vendor_key", "_whse_key", "weekly_lead_time"]
)

conflicting_lt = _lt.loc[
    _lt.duplicated(["_vendor_key", "_whse_key"], keep=False),
    ["Vendor", "whse", "weekly_lead_time"],
].sort_values(["Vendor", "whse", "weekly_lead_time"])

if not conflicting_lt.empty:
    display(conflicting_lt)
    raise ValueError(
        "Conflicting lead-time mappings found for the same Vendor × Whse."
    )

dual1 = _dual.merge(
    _lt[["_vendor_key", "_whse_key", "weekly_lead_time"]],
    on=["_vendor_key", "_whse_key"],
    how="left",
    validate="m:1",
    indicator=True,
)

unmatched = dual1.loc[
    dual1["_merge"].eq("left_only"),
    ["Vendor", "Whse", "VNAME", "Secondary Vendor Number"],
].drop_duplicates().sort_values(["Vendor", "Whse"])

print(
    f"Lead-time matches: {dual1['_merge'].eq('both').sum():,}"
    f"/{len(dual1):,} rows"
)

if not unmatched.empty:
    print("Vendor × Whse pairs without a lead-time mapping:")
    display(unmatched)

dual1["ETD"] = pd.to_datetime(dual1["ETD"], errors="coerce")
dual1["ETA"] = dual1["ETD"] + pd.to_timedelta(
    dual1["weekly_lead_time"] * 7,
    unit="D",
)

dual1 = dual1.drop(columns=["_vendor_key", "_whse_key", "_merge"])

# PDT ETA (SUPPLY - DEMAND)

In [ ]:
query4 = f"""


    SELECT
        PDT.[spdItem] AS [Item #],
        PDT.[spdWarehouse] AS Whse,
        PDT.[spdBeginingBalance] AS [ON HAND],
        PDT.[spdFirmDemands] AS [FIRM DEMANDS],
        PDT.[spdNetForecast] AS [NET FCST],
        PDT.[spdFirmProduction] AS [FIRM PROD],
        PDT.[spdInTransitTransferIn] AS [INTRANSIT TRF IN],
        PDT.[spdOnOrderTransferIn] AS [ON ORDER TRF IN],
        PDT.[spdPlannedTransferIn] AS [PLANNED TRF IN],
        PDT.[spdPlannedTransferOut] AS [PLANNED TRF OUT],
        PDT.[spdPlannedProduction] AS [PLANNED PROD],
        PDT.[spdFirmPurchaseOrders] AS [FIRM POS],
        PDT.[spdPlannedPurchaseOrders] AS [PLANNED POS],
        PDT.[spdFirmTransferOut] AS [FIRM TRF OUT],
        PDT.[spdPromotionalLift] AS [PROMOTIONAL LIFT], 
        PDT.[spdShippableInventory] AS [SHIPPABLE INV],
        PDT.[spdSafetyStock] AS [SAFETY STK],
        CAST(PDT.[dtec] AS date) AS [Snapshot Date],
        PDT.[spdWeekEnding] AS [Due Date],
        info.[Collective Class],
        info.[AFI Finance Division],
        info.[Primary Vendor],    
        info.[Current Status],
        info.[Future Status],    
        info.[Hold Buy Code] AS HB
    FROM [Wholesale_DemandPlanning_AFI].[SupplyPlanDetail] PDT    
    LEFT JOIN [PowerBI_SupplyChain].[CurrentProductDetails] info 
        ON info.[Item SKU] = PDT.[spdItem]
    WHERE CAST(PDT.[dtec] AS date) = (
          SELECT MAX(CAST([dtec] AS date)) 
          FROM [Wholesale_DemandPlanning_AFI].[SupplyPlanDetail]
      )
    ORDER BY PDT.[spdWeekEnding], PDT.[spdItem]

"""

pdt = pd.read_sql(query4, engine)
pdt

In [ ]:
# Map PSW dual-source PO onto PDT by Item x Whse x Due Date (ETA Saturday).
# Quantities are summed; VNAME is concatenated so PDT row count does not grow.
def _key_text(values):
    return (
        values.astype("string")
        .str.strip()
        .str.replace(r"\.0+$", "", regex=True)
    )


def _join_unique(values):
    _vals = values.dropna().astype(str).str.strip()
    return ", ".join(v for v in dict.fromkeys(_vals) if v)


_dual_map = dual1[["Item", "Whse", "ETA", "VNAME", "PQty", "FQty"]].copy()
_dual_map["_item_key"] = _key_text(_dual_map["Item"])
_dual_map["_whse_key"] = _key_text(_dual_map["Whse"])
_dual_map["_eta_key"] = pd.to_datetime(_dual_map["ETA"], errors="coerce").dt.normalize()
for _q in ("PQty", "FQty"):
    _dual_map[_q] = pd.to_numeric(_dual_map[_q], errors="coerce").fillna(0)
_dual_map = _dual_map.dropna(subset=["_eta_key"])

_dual_agg = (
    _dual_map.groupby(["_item_key", "_whse_key", "_eta_key"], as_index=False)
    .agg(
        planned_po=("PQty", "sum"),
        firm_po=("FQty", "sum"),
        vname=("VNAME", _join_unique),
    )
    .rename(
        columns={
            "planned_po": "PLANNED PO + Vendor (dual1)",
            "firm_po": "FIRM PO + Vendor (dual1)",
            "vname": "VNAME (dual1)",
        }
    )
)

_rows_before = len(pdt)
pdt["_item_key"] = _key_text(pdt["Item #"])
pdt["_whse_key"] = _key_text(pdt["Whse"])
pdt["_eta_key"] = pd.to_datetime(pdt["Due Date"], errors="coerce").dt.normalize()

pdt = pdt.merge(
    _dual_agg,
    on=["_item_key", "_whse_key", "_eta_key"],
    how="left",
    validate="m:1",
)

for _c in ("PLANNED PO + Vendor (dual1)", "FIRM PO + Vendor (dual1)"):
    pdt[_c] = pd.to_numeric(pdt[_c], errors="coerce").fillna(0)
pdt["VNAME (dual1)"] = pdt["VNAME (dual1)"].fillna("").astype(str)

_unmapped = _dual_agg.merge(
    pdt[["_item_key", "_whse_key", "_eta_key"]].drop_duplicates(),
    on=["_item_key", "_whse_key", "_eta_key"],
    how="left",
    indicator=True,
)
_unmapped = _unmapped.loc[_unmapped["_merge"].eq("left_only")]
print(f"PDT rows: {_rows_before:,} -> {len(pdt):,}")
print(f"dual1 keys mapped: {len(_dual_agg) - len(_unmapped):,}/{len(_dual_agg):,}")
if not _unmapped.empty:
    display(
        _unmapped[
            [
                "_item_key",
                "_whse_key",
                "_eta_key",
                "VNAME (dual1)",
                "PLANNED PO + Vendor (dual1)",
                "FIRM PO + Vendor (dual1)",
            ]
        ].head(20)
    )

# One VNAME per Item x Whse so the week pivot does not split dual-PO weeks onto a second identity.
pdt["VNAME (dual1)"] = pdt.groupby(["Item #", "Whse"], sort=False)["VNAME (dual1)"].transform(_join_unique)

pdt = pdt.drop(columns=["_item_key", "_whse_key", "_eta_key"])
pdt


In [ ]:
# Melt SI inputs only. SS% and SI-SS are derived after SI roll in EXPORT (not Data types).
pdt1_melted = pdt.melt(
    id_vars=['Item #', 'Whse', 'Snapshot Date', 'Due Date', 'Collective Class', 'AFI Finance Division',
       'Primary Vendor', 'Current Status', 'Future Status', 'HB', 'VNAME (dual1)'],
    value_vars=['ON HAND', 'FIRM DEMANDS', 'NET FCST',
       'FIRM PROD', 'INTRANSIT TRF IN', 'ON ORDER TRF IN', 'PLANNED TRF IN',
       'PLANNED TRF OUT', 'PLANNED PROD', 'FIRM POS', 'PLANNED POS',
       'FIRM TRF OUT', 'PROMOTIONAL LIFT', 'SHIPPABLE INV', 'SAFETY STK',
       'PLANNED PO + Vendor (dual1)', 'FIRM PO + Vendor (dual1)'],
    var_name='Data type',
    value_name='Value'
)


In [ ]:
pdt_pivot = pdt1_melted.pivot_table(index=['Item #', 'Whse','Data type','Snapshot Date', 'Collective Class', 
           'AFI Finance Division', 'Primary Vendor', 'Current Status',
              'Future Status', 'HB'],
    columns='Due Date',
    values='Value',
    aggfunc='sum'
).reset_index()

pdt_pivot.fillna(0, inplace=True)
_vname_whse = (
    pdt1_melted.assign(_vn=pdt1_melted['VNAME (dual1)'].astype(str).str.strip())
    .loc[lambda d: d['_vn'].ne('')]
    .groupby(['Item #', 'Whse'], sort=False)['_vn']
    .agg(lambda s: ', '.join(dict.fromkeys(s)))
    .rename('VNAME (dual1)')
    .reset_index()
)
pdt_pivot = pdt_pivot.merge(_vname_whse, on=['Item #', 'Whse'], how='left')
pdt_pivot['VNAME (dual1)'] = pdt_pivot['VNAME (dual1)'].fillna('').astype(str).str.strip()
pdt_pivot


In [ ]:
#Map Prod — strip keys and unique ItemID so empty Prod/Series is a true miss, not a dup explode.
_item_map = item_profile[['ItemID', 'ProdResourceID', 'Series']].copy()
_item_map['ItemID'] = _item_map['ItemID'].astype(str).str.strip()
_item_map = _item_map.drop_duplicates(subset=['ItemID'], keep='first')
pdt_pivot['Item #'] = pdt_pivot['Item #'].astype(str).str.strip()
pdt_full = pdt_pivot.merge(
    _item_map,
    left_on='Item #',
    right_on='ItemID',
    how='left',
)
pdt_full.drop(columns=['ItemID'], inplace=True)
# Fill identity/profile strings only — do not turn week NaNs into ''.
_str_cols = [
    "ProdResourceID", "Series", "Item #", "Whse", "Data type", "Snapshot Date",
    "Collective Class", "AFI Finance Division", "Primary Vendor",
    "Current Status", "Future Status", "HB", "VNAME (dual1)",
]
for col in _str_cols:
    if col in pdt_full.columns:
        pdt_full[col] = pdt_full[col].fillna("").astype(str)

# Move profile columns to the front (keep names aligned with values).
prod_col = pdt_full.pop('ProdResourceID')
series_col = pdt_full.pop('Series')
pdt_full.insert(0, 'ProdResourceID', prod_col)
pdt_full.insert(1, 'Series', series_col)
pdt_full





In [ ]:
# Map MB Code (Short version)
mb_map = mb_code[["SKU", "Vendor_code", "Location", "MB"]].copy()
for col in ("SKU", "Vendor_code", "Location"):
    mb_map[col] = mb_map[col].astype(str).str.strip()
mb_map["MB"] = mb_map["MB"].fillna("").astype(str).str.strip()
mb_map = mb_map.drop_duplicates(subset=["SKU", "Vendor_code", "Location"])

left = pdt_full.copy()
left["_sku"] = left["Item #"].astype(str).str.strip()
left["_vnd"] = left["Primary Vendor"].astype(str).str.strip()
left["_loc"] = left["Whse"].astype(str).str.strip()

# Map MB (by item+vendor+whse)
pdt_full2 = left.merge(
    mb_map.rename(columns={"SKU": "_sku", "Vendor_code": "_vnd", "Location": "_loc"}),
    on=["_sku", "_vnd", "_loc"], how="left"
)

# Fallback: MB theo item+whse nếu chưa map đc
mk = pdt_full2["MB"].isna() | (pdt_full2["MB"] == "")
if mk.any():
    mb_loc = (
        mb_map.drop_duplicates(subset=["SKU", "Location"])
        .rename(columns={"SKU": "_sku", "Location": "_loc", "MB": "_MB_fb"})
        [["_sku", "_loc", "_MB_fb"]]
    )
    _fb = left.loc[mk, ["_sku", "_loc"]].merge(mb_loc, on=["_sku", "_loc"], how="left")
    pdt_full2.loc[mk, "MB"] = _fb["_MB_fb"].values

pdt_full2.drop(columns=["_sku", "_vnd", "_loc"], inplace=True)
pdt_full2["MB"] = pdt_full2["MB"].fillna("")

# Đưa MB sau Primary Vendor
mb_col = pdt_full2.pop("MB")
idx = pdt_full2.columns.get_loc("Primary Vendor")
pdt_full2.insert(idx + 1, "MB", mb_col)
pdt_full2


# PDT ETD (SUPPLY - DEMAND)


In [ ]:
# PDT ETD: one row = Item × Whse × vendor × week; Data Types become columns.
# pdt_full2 is already wide by week, so melt weeks first, then pivot_table.
import re

_DATE_COL_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")
week_cols = [c for c in pdt_full2.columns if _DATE_COL_RE.match(str(c))]
id_vars = [c for c in pdt_full2.columns if c not in week_cols]
if not week_cols:
    raise KeyError("pdt_full2 has no YYYY-MM-DD week columns to melt into ETD.")

# Melt week columns into long format
pdt_etd_long = pdt_full2.melt(
    id_vars=id_vars,
    value_vars=week_cols,
    var_name="ETA",
    value_name="Value",
)
pdt_etd_long["ETA"] = pd.to_datetime(pdt_etd_long["ETA"], errors="coerce")
pdt_etd_long["Value"] = pd.to_numeric(pdt_etd_long["Value"], errors="coerce").fillna(0)

index_cols = [c for c in id_vars if c not in ("Data type", "VNAME (dual1)")]
pdt_etd = pdt_etd_long.pivot_table(
    index=index_cols + ["ETA"],
    columns="Data type",
    values="Value",
    aggfunc="sum",
    fill_value=0,
).reset_index()
_vname_etd = (
    pdt_etd_long.assign(_vn=pdt_etd_long["VNAME (dual1)"].astype(str).str.strip())
    .loc[lambda d: d["_vn"].ne("")]
    .groupby(["Item #", "Whse"], sort=False)["_vn"]
    .agg(lambda s: ", ".join(dict.fromkeys(s)))
    .rename("VNAME (dual1)")
    .reset_index()
)
pdt_etd = pdt_etd.merge(_vname_etd, on=["Item #", "Whse"], how="left")
pdt_etd["VNAME (dual1)"] = pdt_etd["VNAME (dual1)"].fillna("").astype(str)

# Lead time map: Vendor × warehouse → weekly_lead_time (create _lt here so a fresh kernel works).
_lt = vendor_port_lt[["Vendor", "whse", "weekly_lead_time"]].copy()
_lt["Vendor"] = _lt["Vendor"].astype(str).str.strip()
_lt["whse"] = _lt["whse"].astype(str).str.strip()
_lt["weekly_lead_time"] = pd.to_numeric(_lt["weekly_lead_time"], errors="coerce").fillna(0)
_lt = _lt.drop_duplicates(subset=["Vendor", "whse"], keep="first")

# Clean up vendor/warehouse keys for merging
pdt_etd = pdt_etd.copy()
pdt_etd["Primary Vendor"] = pdt_etd["Primary Vendor"].astype(str).str.strip()
pdt_etd["Whse"] = pdt_etd["Whse"].astype(str).str.strip()

# Merge lead time map to ETD
pdt_etd = pdt_etd.merge(
    _lt,
    left_on=["Primary Vendor", "Whse"],
    right_on=["Vendor", "whse"],
    how="left",
)
pdt_etd = pdt_etd.drop(columns=["Vendor", "whse"])

# Calculate ETD
pdt_etd["ETD"] = pdt_etd["ETA"] - pd.to_timedelta(pdt_etd["weekly_lead_time"] * 7, unit="D")
# SS% / SI-SS are derived after SI roll in EXPORT — do not treat them as Data types.

pdt_etd


In [ ]:
pdt_etd_melt = pdt_etd.melt(
    id_vars=['ProdResourceID', 'Series', 'Item #', 'Whse', 'Snapshot Date',
       'Collective Class', 'AFI Finance Division', 'Primary Vendor', 'MB',
       'Current Status', 'Future Status', 'HB', 'VNAME (dual1)', 'ETD'],
    value_vars=['ON HAND', 'FIRM DEMANDS', 'NET FCST',
       'FIRM PROD', 'INTRANSIT TRF IN', 'ON ORDER TRF IN', 'PLANNED TRF IN',
       'PLANNED TRF OUT', 'PLANNED PROD', 'FIRM POS', 'PLANNED POS',
       'FIRM TRF OUT', 'PROMOTIONAL LIFT', 'SHIPPABLE INV', 'SAFETY STK',
       'PLANNED PO + Vendor (dual1)', 'FIRM PO + Vendor (dual1)'],
    var_name='Data type',
    value_name='Value'
)

pdt_etd_melt

In [ ]:
pdt_etd_pivot = pdt_etd_melt.pivot_table(index=['ProdResourceID', 'Series', 'Item #', 'Whse', 'Snapshot Date',
       'Collective Class', 'AFI Finance Division', 'Primary Vendor', 'MB',
       'Current Status', 'Future Status', 'HB', 'Data type'],
    columns='ETD',
    values='Value',
    aggfunc='sum'
).reset_index()
_vname_etd_whse = (
    pdt_etd_melt.assign(_vn=pdt_etd_melt['VNAME (dual1)'].astype(str).str.strip())
    .loc[lambda d: d['_vn'].ne('')]
    .groupby(['Item #', 'Whse'], sort=False)['_vn']
    .agg(lambda s: ', '.join(dict.fromkeys(s)))
    .rename('VNAME (dual1)')
    .reset_index()
)
pdt_etd_pivot = pdt_etd_pivot.merge(_vname_etd_whse, on=['Item #', 'Whse'], how='left')
pdt_etd_pivot['VNAME (dual1)'] = pdt_etd_pivot['VNAME (dual1)'].fillna('').astype(str).str.strip()

# Shape like pdt_full2: ISO week names + fillna(0) on week columns only.
import re
_DATE_COL_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")
pdt_etd_pivot.columns = [str(c) for c in pdt_etd_pivot.columns]
_renamed = {}
for _col in list(pdt_etd_pivot.columns):
    if _DATE_COL_RE.match(_col):
        continue
    _parsed = pd.to_datetime(_col, errors="coerce")
    if pd.notna(_parsed):
        _renamed[_col] = pd.Timestamp(_parsed).strftime("%Y-%m-%d")
pdt_etd_pivot = pdt_etd_pivot.rename(columns=_renamed)
_week_cols = sorted(c for c in pdt_etd_pivot.columns if _DATE_COL_RE.match(str(c)))
for _col in _week_cols:
    pdt_etd_pivot[_col] = pd.to_numeric(pdt_etd_pivot[_col], errors="coerce").fillna(0)
_id_cols = [c for c in pdt_etd_pivot.columns if c not in _week_cols]
pdt_etd_pivot = pdt_etd_pivot[_id_cols + _week_cols]
pdt_etd_pivot


# CALENDAR

In [ ]:
from datetime import date, timedelta

# Thứ 7 gần nhất (week ending) — khớp snapshot onhand / ETD_Week
today = date.today()
days_since_saturday = (today.weekday() - 5) % 7
current_sat = pd.Timestamp(today - timedelta(days=days_since_saturday))

week_ends = pd.date_range(
    start=current_sat - pd.DateOffset(weeks=3),
    end=current_sat + pd.DateOffset(weeks=45),
    freq="W-SAT",
)

week_calendar = pd.DataFrame({
    "Week_Ending": week_ends,
    "Wn": range(-3, -3 + len(week_ends)),  # 0 = thứ 7 hiện tại
})

week_calendar

In [ ]:
# Giả sử vendor_port_lt đã tồn tại, chứa cột ['vendor', 'whse', ...]
# Mục tiêu: tạo một dataframe tuần (week_calendar) cho từng "whse" có trong vendor_port_lt, map tuần với warehouse.

# Lấy tất cả các whse từ vendor_port_lt, loại bỏ trùng lặp:
whse_list = vendor_port_lt["whse"].unique()

# Tạo dataframe kết hợp tuần (week_calendar) và whse - mỗi whse ứng với tất cả các tuần:
import itertools

week_whse_cartesian = pd.DataFrame(
    list(itertools.product(week_calendar["Week_Ending"], whse_list)),
    columns=["Week_Ending", "whse"]
)

# Nếu cần nối thêm các cột của vendor_port_lt theo whse:
week_whse_cartesian = week_whse_cartesian.merge(
    vendor_port_lt.drop_duplicates(subset=["whse"]),
    on="whse",
    how="left"
)
week_whse_cartesian["ETD"] = (
    pd.to_datetime(week_whse_cartesian["Week_Ending"], errors="coerce")
    - pd.to_timedelta(
        pd.to_numeric(week_whse_cartesian["weekly_lead_time"], errors="coerce").fillna(0),
        unit="W",
    )
)

week_whse_cartesian = week_whse_cartesian.rename(columns={'Week_Ending':"ETA"})
week_whse_cartesian


# Lock Week

In [ ]:
def this_sat(d=None):
    d = d or datetime.today()
    sat = d + timedelta((5 - d.weekday()) % 7)
    return sat.date()

sat0 = this_sat()
lockweek = pd.DataFrame({'Week_Saturday': [sat0 + timedelta(weeks=i) for i in range(4)]})
lockweek["Lockweek"] = "x"

# Convert 'ETD' and 'Week_Saturday' columns to datetime for proper merging
week_whse_cartesian['ETD'] = pd.to_datetime(week_whse_cartesian['ETD'], errors='coerce')
lockweek['Week_Saturday'] = pd.to_datetime(lockweek['Week_Saturday'], errors='coerce')

vendor_calendar_lt = week_whse_cartesian.merge(
    lockweek,
    left_on='ETD',
    right_on='Week_Saturday',
    how='left'
)
vendor_calendar_lt = vendor_calendar_lt.drop(columns='Week_Saturday')
vendor_calendar_lt['Lockweek'] = vendor_calendar_lt['Lockweek'].fillna('')
vendor_calendar_lt

# Push to UI

Đoạn này **không** query EDW. PDT (`pdt_full2`, `pdt_etd_pivot`) đã có trong kernel.

Thứ tự chạy:

1. **VALIDATION** — khung PDT đủ chưa
2. **EXPORT** — ghi `data/` cho website (ETA + ETD)
3. **So SI** — `inv_projection` vs PDT `SHIPPABLE INV` trên **ETA và ETD** (không lên UI)
4. **CSV gap** — xuất các dòng lệch (ETA + ETD)

Website chỉ đọc file cache. EXPORT xong thì Refresh trang.


## 1. VALIDATION

Cửa trước EXPORT. Kiểm tra `pdt_full2` (fallback `pdt_full`):

- Có identity: Item, Whse, Vendor, Data type, Prod, Series, MB
- Có cột tuần `YYYY-MM-DD` và đủ Data Types (ON HAND, Demand/Supply, Promo, Safety)
- Prod không bị đảo với Series

Nếu kernel đã có `export_df` từ lần EXPORT trước:

- Tên biến khớp website; không trùng `(item, vendor, whse, variable)`
- Tuần đầu: `SI = opening + Supply − Demand`
- Promo và Safety **có số nhưng không vào SI**
- Sau EXPORT mới: `SI-SS = SI − Safety`, `SS% = SI / Safety × 100`

Thiếu `ss_pct` / `si_ss` → **SKIP** (chưa EXPORT bản mới). **FAIL** → đừng EXPORT.


In [ ]:
# VALIDATION — pdt_full readiness before / with EXPORT.
# Checks identity columns, supported Data Types, duplicates, and SI W0 invariant
# on the frames EXPORT builds (export_df / export_iv). Re-run EXPORT first if needed.

_tol = 0.01
_problems = []
_date_re = __import__('re').compile(r'^\d{4}-\d{2}-\d{2}$')


def _check(label, ok, detail=""):
    _problems.append((label, bool(ok), detail))
    print(f"{'PASS' if ok else 'FAIL'}  {label}{f'  — {detail}' if detail else ''}")


_src_ok = ("pdt_full2" in dir() and pdt_full2 is not None and len(pdt_full2) > 0) or ("pdt_full" in dir() and pdt_full is not None and len(pdt_full) > 0)
_check("pdt_full2 or pdt_full exists", _src_ok)
_src = pdt_full2 if ("pdt_full2" in dir() and pdt_full2 is not None and len(pdt_full2)) else (pdt_full if ("pdt_full" in dir() and pdt_full is not None and len(pdt_full)) else None)

if _src is not None and len(_src):
    _required = ["Item #", "Whse", "Primary Vendor", "Data type", "ProdResourceID", "Series"]
    if "pdt_full2" in dir() and _src is pdt_full2:
        _required = _required + ["MB"]
    _missing = [c for c in _required if c not in _src.columns]
    _check("pdt_full has identity/profile columns", not _missing, f"missing {_missing}")

    _week_like = []
    for _c in _src.columns:
        if _date_re.match(str(_c)):
            _week_like.append(str(_c))
            continue
        _parsed = pd.to_datetime(_c, errors="coerce")
        if pd.notna(_parsed):
            _week_like.append(pd.Timestamp(_parsed).strftime("%Y-%m-%d"))
    _check("pdt_full has weekly date columns", len(_week_like) > 0, f"{len(set(_week_like))} weeks")

    _types = set(_src["Data type"].astype(str).str.strip())
    _needed = {
        "FIRM DEMANDS", "FIRM TRF OUT", "NET FCST", "PLANNED TRF OUT",
        "INTRANSIT TRF IN", "ON ORDER TRF IN", "PLANNED POS", "PLANNED TRF IN",
        "FIRM POS", "FIRM PROD", "PLANNED PROD", "PROMOTIONAL LIFT", "SAFETY STK", "ON HAND",
    }
    _missing_types = sorted(_needed - _types)
    _check("pdt_full has required Data Types", not _missing_types, f"missing {_missing_types}")

    if {"Series", "ProdResourceID"}.issubset(_src.columns):
        _s = _src["Series"].astype(str).str.contains("PROD", case=False, na=False).mean()
        _p = _src["ProdResourceID"].astype(str).str.contains("PROD", case=False, na=False).mean()
        _check(
            "ProdResourceID looks like Prod (not swapped with Series)",
            _p >= _s,
            f"Series PROD-rate {_s:.1%}, ProdResourceID PROD-rate {_p:.1%}",
        )

if "export_df" not in dir() or export_df is None or len(export_df) == 0:
    print("SKIP  export_df checks — run EXPORT cell first")
else:
    _vars = set(export_df["variable"].astype(str))
    _derived_vars = {"ss_pct", "si_ss"}
    try:
        from export_projection import MATRIX_VARS as _matrix_vars
        _expected_vars = set(_matrix_vars)
    except Exception:
        _expected_vars = {
            "firm_demand", "firm_trf_out", "net_fcst", "planned_trf_out",
            "intransit_trf_in", "on_order_trf_in", "planned_pos", "planned_trf_in",
            "firm_pos", "firm_prod", "planned_prod", "promotional_lift", "safety_stock",
            "inv_projection", "ss_pct", "si_ss",
        }
    _core_vars = _expected_vars - _derived_vars
    _missing_core = sorted(_core_vars - _vars)
    _missing_derived = sorted(_derived_vars - _vars)
    _extra = sorted(_vars - _expected_vars)
    _check(
        "export_df variables match website keys",
        not _missing_core and not _extra,
        f"missing {_missing_core}; extra {_extra}",
    )
    if _missing_derived:
        print(
            "SKIP  ss_pct/si_ss on export_df — "
            f"missing {_missing_derived}; run EXPORT (after SI roll), then re-run VALIDATION"
        )
    else:
        _check("export_df includes ss_pct and si_ss", _derived_vars <= _vars)

    _dupes = export_df.duplicated(subset=["item", "vendor", "whse", "variable"]).sum()
    _check("No duplicate (item, vendor, whse, variable)", _dupes == 0, f"{_dupes:,} duplicates")

    _week_cols = [c for c in export_df.columns if _date_re.match(str(c))]
    _w0 = _week_cols[0] if _week_cols else None
    if _w0:
        _idx = ["item", "vendor", "whse"]
        def _plane(var):
            return (
                export_df.loc[export_df["variable"].eq(var), _idx + [_w0]]
                .drop_duplicates(_idx)
                .set_index(_idx)[_w0]
            )
        _demand = sum(_plane(v).reindex(_plane("inv_projection").index).fillna(0) for v in [
            "firm_demand", "firm_trf_out", "net_fcst", "planned_trf_out",
        ])
        _supply = sum(_plane(v).reindex(_plane("inv_projection").index).fillna(0) for v in [
            "intransit_trf_in", "on_order_trf_in", "planned_pos", "planned_trf_in", "firm_pos", "firm_prod", "planned_prod",
        ])
        # Back out opening from SI W0, then verify SI == opening + supply - demand.
        _si = _plane("inv_projection")
        _opening = _si - _supply + _demand
        _rebuilt = _opening + _supply - _demand
        _gap = float((_rebuilt - _si).abs().max()) if len(_si) else 0.0
        _check("W0 SI = opening + SUPPLY - DEMAND", _gap <= _tol, f"max gap {_gap:,.2f}")

        _promo = float(_plane("promotional_lift").abs().sum())
        _safety = float(_plane("safety_stock").abs().sum())
        _check("Promotional lift present but not in SI", _promo > 0 and _gap <= _tol, f"abs sum {_promo:,.0f}")
        _check("Safety stock present but not in SI", _safety > 0 and _gap <= _tol, f"abs sum {_safety:,.0f}")

        if _derived_vars <= _vars:
            _ss = _plane("safety_stock").reindex(_si.index).fillna(0)
            _si_ss = _plane("si_ss").reindex(_si.index).fillna(0)
            _gap_si_ss = float((_si_ss - (_si - _ss)).abs().max()) if len(_si) else 0.0
            _check("W0 SI-SS = SI - Safety", _gap_si_ss <= _tol, f"max gap {_gap_si_ss:,.2f}")
            _pct = _plane("ss_pct").reindex(_si.index)
            _exp_pct = (_si / _ss * 100.0).where(_ss != 0)
            _obs_pct = _pct.where(_ss != 0)
            _pct_diff = (_obs_pct - _exp_pct).abs()
            _gap_pct = float(_pct_diff.max()) if _pct_diff.notna().any() else 0.0
            if _gap_pct != _gap_pct:
                _gap_pct = 0.0
            _check("W0 SS% = SI / Safety × 100", _gap_pct <= _tol, f"max gap {_gap_pct:,.4f}")

if "export_iv" not in dir() or export_iv is None or len(export_iv) == 0:
    print("SKIP  export_iv checks — run EXPORT cell first")
elif "export_df" in dir() and export_df is not None and len(export_df):
    _week_cols = [c for c in export_df.columns if _date_re.match(str(c))]
    _whse_vars = set(export_df["variable"].astype(str))
    _iv_vars = set(export_iv["variable"].astype(str))
    for _col in [
        "firm_demand", "firm_trf_out", "net_fcst", "planned_trf_out",
        "intransit_trf_in", "on_order_trf_in", "planned_pos", "planned_trf_in",
        "firm_pos", "firm_prod", "planned_prod", "promotional_lift", "safety_stock",
    ]:
        if _col not in _whse_vars and _col not in _iv_vars:
            continue
        if _col not in _whse_vars or _col not in _iv_vars:
            _check(
                f"IV total matches whse total: {_col}",
                False,
                f"missing in {'whse' if _col not in _whse_vars else 'iv'}",
            )
            continue
        _whse_total = float(export_df.loc[export_df["variable"].eq(_col), _week_cols].to_numpy().sum())
        _iv_total = float(export_iv.loc[export_iv["variable"].eq(_col), _week_cols].to_numpy().sum())
        _gap = abs(_whse_total - _iv_total)
        _check(f"IV total matches whse total: {_col}", _gap <= _tol, f"gap {_gap:,.2f}")

    _whse_weeks = [str(w) for w in _week_cols]
    _iv_weeks = [str(w) for w in export_iv.columns if _date_re.match(str(w))]
    _check("Both grains share the same week horizon", _whse_weeks == _iv_weeks)

_failed = [label for label, ok, _ in _problems if not ok]
if _failed:
    raise AssertionError(f"{len(_failed)} validation check(s) failed: {_failed}")
print(f"\nAll {len(_problems)} checks passed.")


## 2. EXPORT — ghi cache cho website

Gọi `export_projection()` **hai lần** (công thức SI khác nhau):

| Nguồn | File trong `data/` | Tab |
|---|---|---|
| `pdt_full2` | `inv_projection_matrix*` / `_iv*` | **ETA by Whse** / **ETA** |
| `pdt_etd_pivot` | `inv_projection_matrix_etd*` / `_etd_iv*` | **ETD by Whse** / **ETD** |

Trong helper: đổi tên PDT → key website → roll SI → sau đó **SS%** và **SI-SS**.

- **ETA:** cắt tuần `week_calendar` (`Wn >= 0`), Lock Week từ `vendor_calendar_lt`. `SI = ON HAND(W-1) + Supply − Demand` mỗi tuần.
- **ETD:** cắt từ **W-1** (cùng Saturday với ETA week[0]); không Lock Week. `SI(W-1)` / `SI(W1)` = `SHIPPABLE INV` copy; từ **W2** roll `SI = SHIPPABLE INV(W1) + Supply − Demand`.

Chạy cell này **bạn tự chạy**. Xong thì Refresh trên website.


In [ ]:
# EXPORT — website cache (inv_projection/data/)
# ETA: ON HAND + Supply − Demand every week.
# ETD: clip from ETA W-1; SI copies SHIPPABLE INV on W-1/W1; roll from W2.
# Re-run this cell yourself after building pdt_full2 and pdt_etd_pivot.

from pathlib import Path
import importlib
import sys

EXPORT_ROOT = Path.cwd() if (Path.cwd() / "app.py").exists() else Path.cwd() / "inv_projection"
if not (EXPORT_ROOT / "app.py").exists():
    raise RuntimeError(
        f"Run this notebook from the inv_projection folder (or its parent); cwd={Path.cwd()}"
    )
if str(EXPORT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPORT_ROOT))

import export_projection as _export_projection
importlib.reload(_export_projection)
from export_projection import DATE_COL_RE, export_projection

export_dir = EXPORT_ROOT / "data"
export_dir.mkdir(parents=True, exist_ok=True)

_week_cal = week_calendar if "week_calendar" in dir() else None
_lock_src = vendor_calendar_lt if "vendor_calendar_lt" in dir() else None

if "pdt_full2" in dir() and pdt_full2 is not None and len(pdt_full2):
    _eta_src, _eta_name = pdt_full2, "pdt_full2"
elif "pdt_full" in dir() and pdt_full is not None and len(pdt_full):
    _eta_src, _eta_name = pdt_full, "pdt_full"
else:
    raise NameError("Build pdt_full2 (preferred) or pdt_full before EXPORT.")

export_df, export_iv = export_projection(
    _eta_src,
    stem="inv_projection_matrix",
    label="ETA",
    export_dir=export_dir,
    week_calendar=_week_cal,
    vendor_calendar_lt=_lock_src,
    clip_to_calendar=True,
    include_lock=True,
    source_name=_eta_name,
)

if "pdt_etd_pivot" in dir() and pdt_etd_pivot is not None and len(pdt_etd_pivot):
    _eta_w_minus_1 = sorted(c for c in export_df.columns if DATE_COL_RE.match(str(c)))[0]
    export_etd, export_etd_iv = export_projection(
        pdt_etd_pivot,
        stem="inv_projection_matrix_etd",
        label="ETD",
        export_dir=export_dir,
        week_calendar=_week_cal,
        vendor_calendar_lt=None,
        clip_to_calendar=False,
        include_lock=False,
        source_name="pdt_etd_pivot",
        si_mode="etd",
        clip_from_week=_eta_w_minus_1,
    )
else:
    export_etd = export_etd_iv = None
    print("SKIP ETD export — build pdt_etd_pivot first.")



## 3. So SI website với PDT `SHIPPABLE INV`

Không đẩy UI. So số roll website với PDT, grain **Item × Whse × tuần** (bỏ vendor), **cả ETA và ETD**.

- `si_pdt` = `SHIPPABLE INV` trên PDT wide (`max` theo item×whse — PDT hay copy cùng SI lên mọi vendor)
- `si_out` = `inv_projection` đã roll (`sum` theo vendor)
- `gap` = `si_out − si_pdt`

ETA dùng `pdt_full2` + `export_df` (cột tuần `ETA`). ETD dùng `pdt_etd_pivot` + `export_etd` (cột `ETD`, đã crop từ W-1). `SHIPPABLE INV` trên ETD là PDT SI đã shift theo lead time.

Trên ETD: W-1 và W1 kỳ vọng gap ~ 0 vs PDT SHIPPABLE (copy). Từ W2 website roll nên gap khác 0 là bình thường.

Thiếu `pdt_etd_pivot` / `export_etd` → **SKIP** ETD. Cell in 20 dòng `|gap|` lớn nhất mỗi trục.


In [ ]:
import re

DATE_COL_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")
IDENTITY_COLS = {
    "ProdResourceID", "Series", "Item #", "Whse", "Data type", "Snapshot Date",
    "Collective Class", "AFI Finance Division", "Primary Vendor",
    "Current Status", "Future Status", "HB", "MB",
    "item", "vendor", "whse", "variable",
}


def _iso_week_rename(df: pd.DataFrame) -> pd.DataFrame:
    """Match EXPORT: datetime / Timestamp week labels -> YYYY-MM-DD strings."""
    out = df.copy()
    out.columns = [str(c) for c in out.columns]
    renamed = {}
    for col in out.columns:
        if col in IDENTITY_COLS or DATE_COL_RE.match(col):
            continue
        parsed = pd.to_datetime(col, errors="coerce")
        if pd.notna(parsed):
            renamed[col] = pd.Timestamp(parsed).strftime("%Y-%m-%d")
    return out.rename(columns=renamed)


def _compare_si(pdt_src, export_whse, week_name: str) -> pd.DataFrame:
    """Rolled inv_projection vs PDT SHIPPABLE INV at item×whse × week."""
    grain = ["Item #", "Whse"]
    pdt_src = _iso_week_rename(pdt_src)
    prj = _iso_week_rename(
        export_whse.loc[export_whse["variable"].astype(str).str.strip().eq("inv_projection")]
        .rename(columns={"item": "Item #", "whse": "Whse", "vendor": "Primary Vendor"})
    )

    week_cols = sorted(c for c in prj.columns if DATE_COL_RE.match(str(c)) and c in pdt_src.columns)
    if not week_cols:
        raise KeyError(f"No overlapping week columns after ISO rename ({week_name}).")

    for frame in (pdt_src, prj):
        for col in grain:
            frame[col] = frame[col].astype(str).str.strip()

    pdt_si = pdt_src.loc[
        pdt_src["Data type"].astype(str).str.strip().eq("SHIPPABLE INV"),
        grain + week_cols,
    ].copy()
    prj_si = prj[grain + week_cols].copy()
    for col in week_cols:
        pdt_si[col] = pd.to_numeric(pdt_si[col], errors="coerce").fillna(0)
        prj_si[col] = pd.to_numeric(prj_si[col], errors="coerce").fillna(0)

    # PDT SI is warehouse-level (often copied onto every vendor row) → do not sum.
    # Projection is rolled per vendor route → sum to item × whse.
    w0 = week_cols[0]
    pdt_vendor_split = int((pdt_si.groupby(grain)[w0].nunique() > 1).sum())
    pdt_iw = pdt_si.groupby(grain, as_index=False)[week_cols].max()
    prj_iw = prj_si.groupby(grain, as_index=False)[week_cols].sum()

    pdt_long = pdt_iw.melt(id_vars=grain, value_vars=week_cols, var_name=week_name, value_name="si_pdt")
    prj_long = prj_iw.melt(id_vars=grain, value_vars=week_cols, var_name=week_name, value_name="si_out")

    out = pdt_long.merge(prj_long, on=grain + [week_name], how="inner")
    out["gap"] = out["si_out"] - out["si_pdt"]

    if "MB" in pdt_src.columns:
        mb_iw = pdt_src[grain + ["MB"]].copy()
        mb_iw["MB"] = mb_iw["MB"].fillna("").astype(str).str.strip()
        mb_iw = (
            mb_iw.loc[mb_iw["MB"].ne("")]
            .drop_duplicates()
            .groupby(grain, as_index=False)["MB"]
            .agg(lambda s: "|".join(sorted(set(s.astype(str)))))
        )
        out = out.merge(mb_iw, on=grain, how="left")
    if "MB" not in out.columns:
        out["MB"] = ""
    else:
        out["MB"] = out["MB"].fillna("")
    out = out[["Item #", "Whse", "MB", week_name, "si_pdt", "si_out", "gap"]]

    print(f"--- {week_name} ---")
    print(f"grain: Item # × Whse × {week_name}")
    print("overlap weeks", len(week_cols), "first", week_cols[0], "last", week_cols[-1])
    print("item×whse", out[grain].drop_duplicates().shape[0], "rows", len(out))
    print(f"PDT SI differs by vendor at item×whse (W0):", pdt_vendor_split)
    print(f"max |gap| by {week_name}:\n", out.groupby(week_name)["gap"].apply(lambda s: s.abs().max()))
    print("share of rows with |gap|>0.01:", float(out["gap"].abs().gt(0.01).mean()))
    print(out.reindex(out["gap"].abs().sort_values(ascending=False).index).head(20).to_string())
    return out


if "pdt_full2" in dir() and pdt_full2 is not None and len(pdt_full2):
    _eta_pdt = pdt_full2
elif "pdt_full" in dir() and pdt_full is not None and len(pdt_full):
    _eta_pdt = pdt_full
else:
    raise NameError("Build pdt_full2 (preferred) or pdt_full before SI compare.")
if "export_df" not in dir() or export_df is None or len(export_df) == 0:
    raise NameError("Run EXPORT first (export_df).")

si_compare = _compare_si(_eta_pdt, export_df, "ETA")

si_compare_etd = None
_etd_ok = (
    "pdt_etd_pivot" in dir() and pdt_etd_pivot is not None and len(pdt_etd_pivot)
    and "export_etd" in dir() and export_etd is not None and len(export_etd)
)
if _etd_ok:
    si_compare_etd = _compare_si(pdt_etd_pivot, export_etd, "ETD")
else:
    print("SKIP ETD SI compare — need pdt_etd_pivot and export_etd (run PDT ETD + EXPORT).")

si_compare.reindex(si_compare["gap"].abs().sort_values(ascending=False).index).head(20)

## 4. Xuất CSV gap

Không đẩy UI. Lọc `|gap| > 0.01`, ghi:

- `data/si_compare_gap.csv` / `si_compare_gap_items.csv` — **ETA**
- `data/si_compare_gap_etd.csv` / `si_compare_gap_items_etd.csv` — **ETD** (nếu đã chạy §3 ETD)

Clipboard chỉ copy `gap_items` ETA để dán Excel.


In [ ]:
# Export SI projection vs PDT SHIPPABLE INV (si_compare / si_compare_etd from the previous cell).
# gap = inv_projection (rolled) − SHIPPABLE INV (PDT wide, max at item×whse).
from pathlib import Path

if "si_compare" not in dir() or si_compare is None or len(si_compare) == 0:
    raise NameError("Run the previous cell first to build si_compare.")

if "export_dir" in dir() and export_dir is not None:
    _gap_dir = Path(export_dir)
else:
    _gap_dir = Path.cwd() / "data" if (Path.cwd() / "app.py").exists() else Path.cwd() / "Inventory Projection" / "data"
_gap_dir.mkdir(parents=True, exist_ok=True)

from datetime import datetime


def _to_csv_unlocked(df, path):
    """Write CSV; if Excel/OneDrive locks the file, use a timestamped sibling."""
    path = Path(path)
    try:
        df.to_csv(path, index=False)
        return path
    except PermissionError:
        alt = path.with_name(f"{path.stem}_{datetime.now().strftime('%Y%m%d_%H%M%S')}{path.suffix}")
        df.to_csv(alt, index=False)
        print(f"Could not overwrite {path.name} (close it in Excel?). Wrote {alt.name} instead.")
        return alt


def _write_si_gap_files(compare, week_col, weeks_name, items_name):
    gap_rows = compare.loc[compare["gap"].abs().gt(0.01)].copy()
    gap_rows = gap_rows.reindex(gap_rows["gap"].abs().sort_values(ascending=False).index)

    _tmp = compare.assign(_abs=compare["gap"].abs())
    gap_items = (
        _tmp.groupby(["Item #", "Whse"], as_index=False)
        .agg(
            n_weeks_gap=("_abs", lambda s: int(s.gt(0.01).sum())),
            max_abs_gap=("_abs", "max"),
            min_gap=("gap", "min"),
            max_gap=("gap", "max"),
        )
    )
    gap_items = gap_items.loc[gap_items["n_weeks_gap"].gt(0)].sort_values("max_abs_gap", ascending=False)

    w0 = sorted(compare[week_col].astype(str).unique())[0]
    w0_vals = compare.loc[compare[week_col].astype(str).eq(w0), ["Item #", "Whse", "si_pdt", "si_out", "gap"]].rename(
        columns={"si_pdt": "si_pdt_w0", "si_out": "si_out_w0", "gap": "gap_w0"}
    )
    gap_items = gap_items.merge(w0_vals, on=["Item #", "Whse"], how="left")

    if "MB" not in compare.columns or compare["MB"].eq("").all():
        _mb_src = None
        if week_col == "ETD" and "pdt_etd_pivot" in dir() and pdt_etd_pivot is not None and "MB" in pdt_etd_pivot.columns:
            _mb_src = pdt_etd_pivot
        elif "pdt_full2" in dir() and pdt_full2 is not None and "MB" in pdt_full2.columns:
            _mb_src = pdt_full2
        if _mb_src is not None:
            _mb = _mb_src[["Item #", "Whse", "MB"]].copy()
            _mb["Item #"] = _mb["Item #"].astype(str).str.strip()
            _mb["Whse"] = _mb["Whse"].astype(str).str.strip()
            _mb["MB"] = _mb["MB"].fillna("").astype(str).str.strip()
            _mb = (
                _mb.loc[_mb["MB"].ne("")]
                .drop_duplicates()
                .groupby(["Item #", "Whse"], as_index=False)["MB"]
                .agg(lambda s: "|".join(sorted(set(s.astype(str)))))
            )
            gap_rows = gap_rows.drop(columns=["MB"], errors="ignore").merge(_mb, on=["Item #", "Whse"], how="left")
            gap_items = gap_items.merge(_mb, on=["Item #", "Whse"], how="left")
    else:
        _mb = compare[["Item #", "Whse", "MB"]].drop_duplicates()
        gap_rows = gap_rows.drop(columns=["MB"], errors="ignore").merge(_mb, on=["Item #", "Whse"], how="left")
        gap_items = gap_items.merge(_mb, on=["Item #", "Whse"], how="left")

    if "MB" not in gap_rows.columns:
        gap_rows["MB"] = ""
    if "MB" not in gap_items.columns:
        gap_items["MB"] = ""
    gap_rows["MB"] = gap_rows["MB"].fillna("")
    gap_items["MB"] = gap_items["MB"].fillna("")
    gap_rows = gap_rows[["Item #", "Whse", "MB", week_col, "si_pdt", "si_out", "gap"]]
    gap_items = gap_items[
        ["Item #", "Whse", "MB", "n_weeks_gap", "max_abs_gap", "min_gap", "max_gap", "si_pdt_w0", "si_out_w0", "gap_w0"]
    ]

    weeks_path = _to_csv_unlocked(gap_rows, _gap_dir / weeks_name)
    items_path = _to_csv_unlocked(gap_items, _gap_dir / items_name)
    print(f"--- {week_col} ---")
    print(f"grain: Item # × Whse  |  weeks: Item # × Whse × {week_col}")
    print("item×whse with |gap|>0.01:", len(gap_items), "of", compare[["Item #", "Whse"]].drop_duplicates().shape[0])
    print("week rows with |gap|>0.01:", len(gap_rows))
    print("W0:", w0)
    print("items CSV:", items_path)
    print("weeks CSV:", weeks_path)
    return gap_items, gap_rows


gap_items, gap_rows = _write_si_gap_files(
    si_compare, "ETA", "si_compare_gap.csv", "si_compare_gap_items.csv"
)
gap_items.to_clipboard(index=False)

if "si_compare_etd" in dir() and si_compare_etd is not None and len(si_compare_etd):
    gap_items_etd, gap_rows_etd = _write_si_gap_files(
        si_compare_etd, "ETD", "si_compare_gap_etd.csv", "si_compare_gap_items_etd.csv"
    )
else:
    print("SKIP ETD gap CSV — run §3 with pdt_etd_pivot + export_etd first.")

